###  Tutorial: Error Handling and Response Validation


In this tutorial, we'll learn how to add proper error handling to agents so they
don't crash when things go wrong. We'll start with a basic agent and upgrade it
with robust error handling.

Learning Objectives:
- Handle common agent errors gracefully
- Add try/catch blocks to agent tools
- Create fallback responses when tools fail
- Build agents that never crash

In [ ]:
# Install required packages
# pip install pydantic-ai openai python-dotenv

import os
from typing import List
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

load_dotenv()

### Basic Agent (That Breaks)

Let's start with a simple agent that can crash.

In [ ]:
basic_agent = Agent(openrouter_model)

In [ ]:
@basic_agent.tool
async def breakable_search(ctx: RunContext[None], query: str) -> str:
    """Search tool that can fail."""
    import random
    if random.random() < 0.5:  # 50% chance of failure
        raise Exception("Network timeout!")
    return f"Search results for: {query}"

### Adding Try/Catch

Now let's fix it with error handling.

In [ ]:
class RobustResponse(BaseModel):
    answer: str = Field(description="The main answer")
    had_errors: bool = Field(description="Whether there were errors")

robust_agent = Agent(openrouter_model, output_type=RobustResponse)

In [ ]:
@robust_agent.tool
async def safe_search(ctx: RunContext[None], query: str) -> str:
    """Search tool with error handling."""
    try:
        import random
        if random.random() < 0.5:
            raise Exception("Network timeout!")
        return f"🔍 Search results for: {query}"
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return f"🔍 Search unavailable. Using general knowledge about: {query}"

In [ ]:
@robust_agent.tool
async def safe_calculation(ctx: RunContext[None], expression: str) -> str:
    """Calculator with error handling."""
    try:
        # Simple validation
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            raise ValueError("Invalid characters")
        
        result = eval(expression)
        return f"🧮 {expression} = {result}"
    except ZeroDivisionError:
        return f"🧮 Cannot divide by zero in: {expression}"
    except Exception as e:
        return f"🧮 Cannot calculate: {expression} ({e})"

### Simple Error Tracking

Let's add basic error tracking.

In [ ]:
error_count = 0

def track_error(operation: str, error: str):
    """Simple error tracking."""
    global error_count
    error_count += 1
    print(f"📊 Error #{error_count}: {operation} failed - {error}")

In [ ]:
@robust_agent.tool
async def tracked_operation(ctx: RunContext[None], operation: str) -> str:
    """Operation with error tracking."""
    try:
        import random
        if random.random() < 0.3:
            raise Exception(f"{operation} service unavailable")
        
        return f"✅ {operation} completed successfully"
    except Exception as e:
        track_error(operation, str(e))
        return f"⚠️ {operation} failed, using fallback response"

In [ ]:
test_cases = [
    "Search for Python tutorials",
    "Calculate 10 + 5", 
    "Calculate 10 / 0"
]

for test in test_cases:
    print(f"\nTest: {test}")
    try:
        result = await robust_agent.run(test)
        print(f"✅ Success: {result.output.had_errors}")
        print(f"Answer: {result.output.answer[:80]}...")
    except Exception as e:
        print(f"❌ Failed: {e}")